# SUI, suscriptores de EPM en Antioquia, 2021

Abrí `muestras/sui-epm-antioquia-2021-suscriptores.csv`. Primero lo intenté en UTF-8, que es lo que pandas usa si no le digo otra cosa.


In [1]:
from pathlib import Path
import pandas as pd

ruta = Path("..") / "muestras" / "sui-epm-antioquia-2021-suscriptores.csv"
try:
    pd.read_csv(ruta)
except UnicodeDecodeError as error:
    error


In [2]:
sui = pd.read_csv(ruta, encoding="cp1252", dtype=str)
sui.shape


(130, 16)

In [3]:
sui.columns.tolist()


['Departamento',
 'Municipio',
 'Empresa',
 'Variable Calculada',
 'Estrato 1',
 'Estrato 2',
 'Estrato 3',
 'Estrato 4',
 'Estrato 5',
 'Estrato 6',
 'Total Residencial',
 'Industrial',
 'Comercial',
 'Oficial',
 'Otros',
 'Total No Residencial']

In [4]:
sui.tail(8)


,Departamento,Municipio,Empresa,Variable Calculada,Estrato 1,Estrato 2,Estrato 3,Estrato 4,Estrato 5,Estrato 6,Total Residencial,Industrial,Comercial,Oficial,Otros,Total No Residencial
122,ANTIOQUIA,ZARAGOZA,EMPRESAS PÚBLICAS DE MEDELLIN E.S.P.,usuarios Empresa Departamento y Municipio,7397,1777,24,24,ND,ND,9222,31,571,90,5,697
123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,Año,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,Período,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
126,Ubicación,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127,Departamento,ANTIOQUIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,Empresa,EMPRESAS PÚBLICAS DE MEDELLIN E.S.P.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,Reporte a Consultar,Suscriptores,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


El archivo no abre en UTF-8: hay un byte que esa codificación no acepta. Lo leí en cp1252. Pandas me dio 130 filas, pero las últimas no son municipios. Son el pie del reporte: año 2021, período 25, ubicación Total, Antioquia, EPM, variable Suscriptores.


In [5]:
municipios = sui[sui["Departamento"] == "ANTIOQUIA"].copy()
municipios.shape


(123, 16)

In [6]:
municipios["Empresa"].unique()


<ArrowStringArray>
['EMPRESAS PÚBLICAS DE MEDELLIN E.S.P.']
Length: 1, dtype: str

In [7]:
municipios["Variable Calculada"].unique()


<ArrowStringArray>
['usuarios Empresa Departamento y Municipio']
Length: 1, dtype: str

In [8]:
medellin = municipios[municipios["Municipio"] == "MEDELLÍN"].iloc[0]
medellin


Departamento                                            ANTIOQUIA
Municipio                                                MEDELLÍN
Empresa                      EMPRESAS PÚBLICAS DE MEDELLIN E.S.P.
Variable Calculada      usuarios Empresa Departamento y Municipio
Estrato 1                                                  129315
Estrato 2                                                  307598
Estrato 3                                                  258602
Estrato 4                                                  102812
Estrato 5                                                   73005
Estrato 6                                                   39254
Total Residencial                                          910586
Industrial                                                   7200
Comercial                                                   76833
Oficial                                                      1848
Otros                                                         526
Total No R

In [9]:
estratos = ["Estrato 1", "Estrato 2", "Estrato 3", "Estrato 4", "Estrato 5", "Estrato 6"]
resto = ["Industrial", "Comercial", "Oficial", "Otros"]
numeros = medellin[estratos + ["Total Residencial"] + resto + ["Total No Residencial"]].astype(int)
numeros


Estrato 1               129315
Estrato 2               307598
Estrato 3               258602
Estrato 4               102812
Estrato 5                73005
Estrato 6                39254
Total Residencial       910586
Industrial                7200
Comercial                76833
Oficial                   1848
Otros                      526
Total No Residencial     86407
Name: 68, dtype: int64

In [10]:
int(numeros[estratos].sum()), int(numeros["Total Residencial"]), int(numeros[resto].sum()), int(numeros["Total No Residencial"])


(910586, 910586, 86407, 86407)

In [11]:
int(numeros["Total Residencial"] + numeros["Total No Residencial"])


996993

In [12]:
columnas = estratos + ["Total Residencial"] + resto + ["Total No Residencial"]
{columna: int((municipios[columna] == "ND").sum()) for columna in columnas}


{'Estrato 1': 0,
 'Estrato 2': 0,
 'Estrato 3': 0,
 'Estrato 4': 0,
 'Estrato 5': 33,
 'Estrato 6': 42,
 'Total Residencial': 0,
 'Industrial': 0,
 'Comercial': 0,
 'Oficial': 0,
 'Otros': 0,
 'Total No Residencial': 0}

Me quedé con 123 filas cuyo departamento es `ANTIOQUIA`. Todas son de Empresas Públicas de Medellín y la variable es suscriptores. El período 25 del pie es el anual. El 1 sería enero, y ese archivo no es este.

Medellín trae el nombre `MEDELLÍN`, no el código. Estratos 1 a 6: 129.315, 307.598, 258.602, 102.812, 73.005 y 39.254. Sumé los estratos y me dio el total residencial, 910.586. Industrial 7.200, comercial 76.833, oficial 1.848 y otros 526 suman el no residencial, 86.407. Juntos son 996.993 suscriptores.

En las 123 filas encontré 75 celdas `ND`. No las convertí a cero. El grano es un municipio, un año, suscriptores de EPM, ubicación Total. No hay kWh y no hay un cliente. El código lo voy a pegar después, con una tabla puente contra el DIVIPOLA, no con un join suelto por el nombre.
